# Module 1: USDA WASDE Report Extraction

## Learning Objectives

By completing this notebook, you will:
1. Download and parse USDA WASDE (World Agricultural Supply and Demand Estimates) reports
2. Extract global supply and demand data for major agricultural commodities
3. Handle complex table structures with LLM assistance
4. Build comparative analysis across report versions (monthly changes)
5. Create trading signals from USDA forecast revisions

## Prerequisites

- Completed Module 1, Notebook 1 (EIA Extraction)
- Understanding of agricultural commodity markets
- LLM API access (OpenAI or Anthropic)
- Familiarity with supply/demand balance sheets

## Estimated Time: 75-90 minutes

---

## 1. Understanding WASDE Reports

The USDA WASDE report is released monthly (typically around the 12th) and moves agricultural commodity markets significantly.

**What Makes WASDE Unique:**
- Global coverage (US and world supply/demand)
- Forward-looking (current + next season forecasts)
- Multiple commodities (grains, oilseeds, cotton, sugar)
- Complex nested tables

**Why Traders Care:**
- Forecast revisions signal supply/demand imbalances
- Ending stocks-to-use ratio indicates tightness
- Production estimates affect price expectations months ahead

In [ ]:
# Standard library imports
import os
import json
import re
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

# PDF processing
import PyPDF2
import pdfplumber

# LLM imports
import openai
from anthropic import Anthropic

# Configuration
load_dotenv()
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

# API credentials
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize LLM clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

# Create directories
DATA_DIR = Path('data/wasde_reports')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print(f"   Data directory: {DATA_DIR.absolute()}")
print(f"   LLM clients: {'OpenAI' if openai_client else ''} {'Claude' if anthropic_client else ''}")

## 2. Downloading WASDE Reports

USDA publishes WASDE reports as PDFs at:
`https://www.usda.gov/oce/commodity/wasde/wasde{MMYY}.pdf`

**Example:** January 2024 report = `wasde0124.pdf`

**Challenges:**
- Release dates vary (not always same day of month)
- Numbering format changed over time
- Reports are 40-50 pages (much larger than EIA)

In [ ]:
def generate_wasde_url(year: int, month: int) -> str:
    """
    Generate USDA WASDE report URL.
    
    Parameters:
    -----------
    year : int
        Year (4 digits)
    month : int
        Month (1-12)
        
    Returns:
    --------
    str
        URL to WASDE PDF
    """
    # Format: wasdeMMYY.pdf (e.g., wasde0124.pdf for January 2024)
    month_str = f"{month:02d}"
    year_str = f"{year % 100:02d}"  # Last 2 digits
    filename = f"wasde{month_str}{year_str}.pdf"
    
    url = f"https://www.usda.gov/oce/commodity/wasde/{filename}"
    return url

def download_wasde_report(year: int, month: int, save_dir: Path = DATA_DIR) -> Optional[Path]:
    """
    Download USDA WASDE report.
    
    Parameters:
    -----------
    year : int
        Year
    month : int
        Month
    save_dir : Path
        Directory to save PDF
        
    Returns:
    --------
    Optional[Path]
        Path to downloaded file, or None if failed
    """
    url = generate_wasde_url(year, month)
    filename = f"wasde_{year}_{month:02d}.pdf"
    filepath = save_dir / filename
    
    # Check if already exists
    if filepath.exists():
        print(f"✅ Report already exists: {filename}")
        return filepath
    
    try:
        print(f"Downloading {filename}...")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        filepath.write_bytes(response.content)
        print(f"✅ Downloaded: {filename} ({len(response.content) / 1024:.1f} KB)")
        return filepath
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"⚠️  Report not found (404): {year}-{month:02d}")
        else:
            print(f"❌ HTTP error {e.response.status_code}")
        return None
    except Exception as e:
        print(f"❌ Download failed: {str(e)}")
        return None

# Download the most recent report
current_date = datetime.now()
# Try current month first, then previous
print(f"Attempting to download WASDE report for {current_date.strftime('%B %Y')}...\n")
wasde_path = download_wasde_report(current_date.year, current_date.month)

if not wasde_path:
    print(f"\nTrying previous month...")
    prev_month = current_date.month - 1 if current_date.month > 1 else 12
    prev_year = current_date.year if current_date.month > 1 else current_date.year - 1
    wasde_path = download_wasde_report(prev_year, prev_month)

if wasde_path:
    print(f"\n✅ Report ready: {wasde_path.name}")
else:
    print("\n⚠️  Could not download recent report")

## 3. Extracting WASDE Tables

WASDE reports contain numerous tables, one per commodity. Each table shows:
- Supply (beginning stocks, production, imports)
- Demand (domestic use, exports)
- Ending stocks
- Prices

**Key Tables:**
- Table 1: Corn supply and demand
- Table 2: Soybean supply and demand
- Table 3: Wheat supply and demand
- Tables 4-12: Other commodities and world balances

In [ ]:
def extract_wasde_tables(pdf_path: Path) -> Dict[str, Any]:
    """
    Extract all tables from WASDE report.
    
    Parameters:
    -----------
    pdf_path : Path
        Path to WASDE PDF
        
    Returns:
    --------
    dict
        Dictionary with pages, text, and extracted tables
    """
    result = {
        'pages': [],
        'tables': [],
        'full_text': ''
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            print(f"Processing WASDE PDF: {pdf_path.name}")
            print(f"  Total pages: {len(pdf.pages)}")
            
            for i, page in enumerate(pdf.pages):
                # Extract text
                text = page.extract_text()
                result['pages'].append({
                    'page_num': i + 1,
                    'text': text
                })
                result['full_text'] += f"\n--- Page {i+1} ---\n{text}"
                
                # Extract tables
                tables = page.extract_tables()
                for j, table in enumerate(tables):
                    if table and len(table) > 2:  # At least header + 2 data rows
                        result['tables'].append({
                            'page': i + 1,
                            'table_num': j + 1,
                            'data': table
                        })
            
            print(f"  ✅ Extracted {len(result['tables'])} tables")
            print(f"  ✅ Total text length: {len(result['full_text'])} characters")
            
    except Exception as e:
        print(f"❌ Error extracting PDF: {str(e)}")
    
    return result

# Extract tables from WASDE
if wasde_path and wasde_path.exists():
    wasde_extracted = extract_wasde_tables(wasde_path)
    print(f"\nFound {len(wasde_extracted['tables'])} tables in report")
    
    # Show first few table locations
    print("\nFirst 5 tables:")
    for i, table_info in enumerate(wasde_extracted['tables'][:5]):
        print(f"  Table {i+1}: Page {table_info['page']}, {len(table_info['data'])} rows")
else:
    print("⚠️  No WASDE report to extract")
    wasde_extracted = None

## 4. Identifying Commodity Tables with LLM

WASDE tables don't have consistent numbering or titles. We need to identify which table corresponds to which commodity.

**LLM Advantage:** Can read table headers and determine content, even with formatting variations.

In [ ]:
def identify_commodity_table(table_data: List[List[str]]) -> dict:
    """
    Use LLM to identify what commodity a table contains.
    
    Parameters:
    -----------
    table_data : List[List[str]]
        Raw table data
        
    Returns:
    --------
    dict
        Information about the table content
    """
    # Convert table to string representation
    table_str = "\n".join([" | ".join([str(cell) for cell in row]) for row in table_data[:10]])
    
    prompt = f"""
Analyze this table from a USDA WASDE report and identify its content.

Return JSON:
{{
  "commodity": "corn|soybeans|wheat|rice|cotton|sugar|etc",
  "region": "us|world",
  "table_type": "supply_demand|price|area_yield|other",
  "confidence": 0.0-1.0
}}

Table (first 10 rows):
{table_str}
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "You are a USDA report analyst. Return only valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = message.content[0].text
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0]
            return json.loads(response_text.strip())
        
        else:
            return {"error": "No LLM configured"}
            
    except Exception as e:
        return {"error": str(e)}

# Identify all tables
if wasde_extracted and wasde_extracted['tables']:
    print("Identifying table contents with LLM...\n")
    
    table_catalog = []
    
    for i, table_info in enumerate(wasde_extracted['tables'][:10]):  # First 10 tables
        print(f"Analyzing table {i+1}/10...", end=' ')
        
        identification = identify_commodity_table(table_info['data'])
        
        if 'error' not in identification:
            table_catalog.append({
                'page': table_info['page'],
                'table_num': table_info['table_num'],
                'commodity': identification.get('commodity', 'unknown'),
                'region': identification.get('region', 'unknown'),
                'type': identification.get('table_type', 'unknown'),
                'confidence': identification.get('confidence', 0.0)
            })
            print(f"✅ {identification.get('commodity', '?')} ({identification.get('region', '?')})")
        else:
            print(f"❌ Error: {identification['error']}")
    
    # Display catalog
    if table_catalog:
        catalog_df = pd.DataFrame(table_catalog)
        print("\nTable Catalog:")
        print(catalog_df.to_string(index=False))
else:
    print("⚠️  No tables to identify")
    table_catalog = []

### 💡 Exercise 4.1: Find Corn Supply and Demand Table

**Task:** Using the table catalog, locate the US Corn supply and demand table and extract it.

**Requirements:**
1. Filter the catalog for commodity='corn' and region='us'
2. Get the table data from wasde_extracted['tables']
3. Convert to a pandas DataFrame
4. Display the first few rows

**Hints:**
- Match on page and table_num
- First row is typically headers
- Handle merged cells and multi-level headers

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if table_catalog and wasde_extracted:
    # Find corn US table
    corn_tables = [t for t in table_catalog if t['commodity'] == 'corn' and t['region'] == 'us']
    
    if corn_tables:
        corn_table_info = corn_tables[0]  # Take first match
        print(f"Found corn table: Page {corn_table_info['page']}, Table {corn_table_info['table_num']}")
        
        # Get the raw table data
        matching_table = None
        for table in wasde_extracted['tables']:
            if table['page'] == corn_table_info['page'] and table['table_num'] == corn_table_info['table_num']:
                matching_table = table['data']
                break
        
        if matching_table:
            # Convert to DataFrame
            corn_df = pd.DataFrame(matching_table)
            
            # Use first row as headers
            corn_df.columns = corn_df.iloc[0]
            corn_df = corn_df[1:].reset_index(drop=True)
            
            print("\nCorn Supply and Demand Table:")
            print(corn_df.head(10))
            print(f"\nShape: {corn_df.shape}")
        else:
            print("❌ Could not find matching table data")
    else:
        print("⚠️  No corn table found in catalog")
else:
    print("⚠️  No catalog or extracted data available")

## 5. Parsing Supply and Demand Data

Supply and demand tables have a consistent structure:
- Rows: Supply items (production, imports, etc.) and demand items (exports, domestic use)
- Columns: Different time periods (current year, previous year, forecast)

**Key Metrics:**
- **Production**: Total crop size
- **Ending Stocks**: What's left over
- **Stocks-to-Use Ratio**: Ending stocks / total use (tightness indicator)

In [ ]:
def parse_supply_demand_table(table_data: List[List[str]], commodity: str) -> pd.DataFrame:
    """
    Parse WASDE supply and demand table into clean DataFrame.
    
    Parameters:
    -----------
    table_data : List[List[str]]
        Raw table data
    commodity : str
        Commodity name for labeling
        
    Returns:
    --------
    pd.DataFrame
        Cleaned supply and demand data
    """
    df = pd.DataFrame(table_data)
    
    # First row typically contains column headers
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    
    # Clean numeric columns (all except first)
    item_col = df.columns[0]
    for col in df.columns[1:]:
        # Remove commas, handle empty cells
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '').str.replace('--', '').str.strip(),
            errors='coerce'
        )
    
    # Add commodity label
    df['commodity'] = commodity
    
    # Rename first column for consistency
    df = df.rename(columns={item_col: 'item'})
    
    return df

# Example: Parse a corn table if we have one
if 'corn_df' in locals() and corn_df is not None:
    parsed_corn = parse_supply_demand_table(matching_table, 'corn')
    print("Parsed Corn Data:")
    print(parsed_corn.head(10))
else:
    print("⚠️  No corn table to parse")

## 6. Extracting Key Metrics with LLM

For trading signals, we need specific numbers:
- Current production estimate
- Ending stocks
- Changes from previous month

LLMs can help extract these even when table formats vary.

In [ ]:
def extract_key_metrics(table_text: str, commodity: str) -> dict:
    """
    Use LLM to extract key trading metrics from supply/demand table.
    
    Parameters:
    -----------
    table_text : str
        String representation of table
    commodity : str
        Commodity name
        
    Returns:
    --------
    dict
        Key metrics for trading analysis
    """
    prompt = f"""
Extract key metrics from this USDA {commodity} supply and demand table.

Return JSON:
{{
  "current_year_production": <number>,
  "current_year_ending_stocks": <number>,
  "previous_year_production": <number>,
  "previous_year_ending_stocks": <number>,
  "total_use_current": <number>,
  "stocks_to_use_ratio": <calculated as ending_stocks/total_use>,
  "production_change_yoy": <current - previous, in same units>,
  "units": "million bushels|million metric tons|etc"
}}

Table:
{table_text}
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "Extract numerical data accurately. Return valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = message.content[0].text
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0]
            return json.loads(response_text.strip())
        
        else:
            return {"error": "No LLM configured"}
    
    except Exception as e:
        return {"error": str(e)}

# Extract metrics from corn table
if 'parsed_corn' in locals() and parsed_corn is not None:
    # Convert to text representation
    corn_text = parsed_corn.to_string(index=False)
    
    print("Extracting key metrics from corn table...\n")
    corn_metrics = extract_key_metrics(corn_text, 'corn')
    
    print("Extracted Corn Metrics:")
    print(json.dumps(corn_metrics, indent=2))
    
    # Calculate additional signals
    if 'stocks_to_use_ratio' in corn_metrics:
        ratio = corn_metrics['stocks_to_use_ratio']
        print(f"\nStocks-to-Use Ratio: {ratio:.2%}")
        if ratio < 0.10:
            print("  ⚠️  TIGHT SUPPLY - Bullish signal")
        elif ratio > 0.20:
            print("  ⚠️  AMPLE SUPPLY - Bearish signal")
        else:
            print("  ✅ BALANCED SUPPLY")
else:
    print("⚠️  No parsed corn data available")

### 💡 Exercise 6.1: Multi-Commodity Extraction

**Task:** Extract key metrics for corn, soybeans, and wheat.

**Requirements:**
1. Find tables for all three commodities in the catalog
2. Parse each table's supply and demand data
3. Extract key metrics using the LLM function
4. Create a comparison DataFrame showing:
   - Commodity name
   - Stocks-to-use ratio
   - Production change YoY
5. Identify which commodity has the tightest supply

**Expected Output:** Comparison table and supply tightness ranking

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if table_catalog and wasde_extracted:
    target_commodities = ['corn', 'soybeans', 'wheat']
    commodity_metrics = []
    
    for commodity in target_commodities:
        print(f"\nProcessing {commodity}...")
        
        # Find table
        tables = [t for t in table_catalog if t['commodity'] == commodity and t['region'] == 'us']
        
        if tables:
            table_info = tables[0]
            
            # Get raw data
            for table in wasde_extracted['tables']:
                if table['page'] == table_info['page'] and table['table_num'] == table_info['table_num']:
                    # Parse and extract
                    parsed = parse_supply_demand_table(table['data'], commodity)
                    table_text = parsed.to_string(index=False)
                    metrics = extract_key_metrics(table_text, commodity)
                    
                    if 'error' not in metrics:
                        commodity_metrics.append({
                            'commodity': commodity,
                            'stocks_to_use': metrics.get('stocks_to_use_ratio', None),
                            'production_change_yoy': metrics.get('production_change_yoy', None),
                            'ending_stocks': metrics.get('current_year_ending_stocks', None)
                        })
                        print(f"  ✅ Extracted metrics for {commodity}")
                    break
        else:
            print(f"  ⚠️  No table found for {commodity}")
    
    if commodity_metrics:
        comparison_df = pd.DataFrame(commodity_metrics)
        comparison_df = comparison_df.sort_values('stocks_to_use')
        
        print("\n" + "="*60)
        print("Commodity Supply Comparison")
        print("="*60)
        print(comparison_df.to_string(index=False))
        
        # Identify tightest supply
        tightest = comparison_df.iloc[0]
        print(f"\n⚠️  TIGHTEST SUPPLY: {tightest['commodity'].upper()}")
        print(f"   Stocks-to-Use: {tightest['stocks_to_use']:.2%}")
    else:
        print("\n⚠️  No metrics extracted")
else:
    print("⚠️  No data available")

## 7. Comparing Reports for Trading Signals

The real trading signal comes from changes between monthly reports.

**Example Signals:**
- Production cut = Bullish
- Ending stocks increase = Bearish
- Demand upgrade = Bullish

To generate these, we need to compare current vs. previous month's report.

In [ ]:
def compare_wasde_reports(current_metrics: dict, previous_metrics: dict) -> dict:
    """
    Compare two WASDE reports to generate trading signals.
    
    Parameters:
    -----------
    current_metrics : dict
        Metrics from current month
    previous_metrics : dict
        Metrics from previous month
        
    Returns:
    --------
    dict
        Changes and trading signals
    """
    signals = {
        'production_revision': 0,
        'stocks_revision': 0,
        'stocks_to_use_change': 0,
        'overall_signal': 'neutral'
    }
    
    # Calculate changes
    if 'current_year_production' in current_metrics and 'current_year_production' in previous_metrics:
        signals['production_revision'] = (
            current_metrics['current_year_production'] - 
            previous_metrics['current_year_production']
        )
    
    if 'current_year_ending_stocks' in current_metrics and 'current_year_ending_stocks' in previous_metrics:
        signals['stocks_revision'] = (
            current_metrics['current_year_ending_stocks'] - 
            previous_metrics['current_year_ending_stocks']
        )
    
    if 'stocks_to_use_ratio' in current_metrics and 'stocks_to_use_ratio' in previous_metrics:
        signals['stocks_to_use_change'] = (
            current_metrics['stocks_to_use_ratio'] - 
            previous_metrics['stocks_to_use_ratio']
        )
    
    # Determine overall signal
    # Bearish if stocks increase, bullish if stocks decrease
    if signals['stocks_to_use_change'] > 0.01:
        signals['overall_signal'] = 'bearish'
    elif signals['stocks_to_use_change'] < -0.01:
        signals['overall_signal'] = 'bullish'
    
    return signals

# Example comparison (would need two months of data)
print("Trading Signal Generation:")
print("\nTo generate signals, we need:")
print("  1. Current month WASDE data")
print("  2. Previous month WASDE data")
print("  3. Compare key metrics")
print("\nExample signals:")
print("  - Stocks-to-use drops 2% → BULLISH (tighter supply)")
print("  - Production cut 50 million bushels → BULLISH")
print("  - Ending stocks increase → BEARISH (more supply)")

### 💡 Exercise 7.1: Download and Compare Two Reports

**Task:** Download the previous month's WASDE report and compare corn metrics.

**Requirements:**
1. Download the report from one month prior
2. Extract the corn table from both reports
3. Extract key metrics from both
4. Use `compare_wasde_reports()` to generate signals
5. Print a trading recommendation

**This exercise requires actual data access - implement as much as possible with available reports.**

In [ ]:
# YOUR CODE HERE
# ---------------





## Summary

### Key Takeaways

1. **WASDE Reports Are Complex** - Multiple commodities, nested tables, varying formats

2. **LLMs Excel at Table Identification** - Can determine content even with format variations

3. **Supply/Demand Balances Drive Prices** - Focus on ending stocks and stocks-to-use ratios

4. **Changes Matter More Than Levels** - Monthly revisions generate trading signals

5. **Multi-Commodity Analysis Finds Opportunities** - Compare relative tightness across crops

### Skills Gained

✅ Downloading USDA WASDE reports  
✅ Extracting tables from complex PDFs  
✅ Using LLMs for table identification  
✅ Parsing supply and demand data  
✅ Calculating stocks-to-use ratios  
✅ Generating trading signals from USDA revisions  

### What's Next

In **Module 2, Notebook 1** (`01_eia_knowledge_base.ipynb`), we'll build a RAG system to answer questions about commodity reports using:
- Vector databases for report storage
- Semantic search for relevant sections
- LLM-powered Q&A over historical data

### Additional Resources

- **USDA WASDE Reports:** https://www.usda.gov/oce/commodity/wasde
- **Agricultural Market Analysis:** https://www.nass.usda.gov/
- **Commodity Futures Trading:** Understanding supply/demand fundamentals

---

**Next:** [Module 2 - RAG Research Systems](../../module_02_rag_research/notebooks/01_eia_knowledge_base.ipynb)